# Create databases

This will create databases from root schema for testing clearscape analytics

Database hierarchy is
CS01
CS01_RAW
CS01_RAW_VIEW
CS01_TARGET
CS01_TARGET_VIEW

In [5]:
import teradatasql
import yaml
import sys
import os
import sqlparse
import pandas as pd



In [6]:
# get home directory
from os.path import expanduser
import pprint
home = expanduser("~")
# assume project is called dbt_clearscape
project = 'dbt_clearscape'
# read config file
with open(home + '/.dbt/profiles.yml') as file:
    config = yaml.load(file, Loader=yaml.FullLoader)
# get project config
if project not in config:
    print('project not found in config file')
project_config = config[project]
# get target
target = project_config['target']
# get connection details
project_connection = project_config['outputs'][target]
#pprint.pprint(project_connection)


Connect to database

In [20]:
# connect to teradata
conn = teradatasql.connect(host=project_connection['server'], user=project_connection['username'], password=project_connection['password'])    

# Create all the database

The databases are

- CS01
    - CS01_RAW
    - CS01_RAW_VIEW
    - CS01_TARGET
    - CS01_TARGET_VIEW
    - CS01_TEMP
        - CS01_TEMP_DATA
    - CS01_BKEY
        - CS01_BKEY_DATA
        - CS01_BKEY_INP
        - CS01_BKEY_OUT
        - CS01_BKEY_ACCESS
    - CS01_CORE
        - CS01_CORE_DATA
        - CS01_CORE_INP
        - CS01_CORE_OUT
        - CS01_CORE_ACCESS
    - CS01_CLEARSCAPE
        - CS01_CLEARSCAPE_DATA
        - CS01_CLEARSCAPE_INP
        - CS01_CLEARSCAPE_OUT
        - CS01_CLEARSCAPE_ACCESS
  



In [22]:
# TODO - setup parameterised db config
# create database hierarchy as below    
#- CS01
#    - CS01_RAW
#    - CS01_RAW_VIEW
#    - CS01_TARGET
#    - CS01_TARGET_VIEW
#    - CS01_TEMP
#        - CS01_TEMP_DATA
#    - CS01_BKEY
#        - CS01_BKEY_DATA
#        - CS01_BKEY_INP
#        - CS01_BKEY_OUT
#        - CS01_BKEY_ACCESS
#    - CS01_CORE
#        - CS01_CORE_DATA
#        - CS01_CORE_INP
#        - CS01_CORE_OUT
#        - CS01_CORE_ACCESS
#    - CS01_CLEARSCAPE
#        - CS01_CLEARSCAPE_DATA
#        - CS01_CLEARSCAPE_INP
#        - CS01_CLEARSCAPE_OUT
#        - CS01_CLEARSCAPE_ACCESS
  
db_list = {}
db_list ['CS01'] = {'owner': project_connection['username'],'space':0}
db_list ['CS01_RAW'] = {'owner': 'CS01','space':0}
db_list ['CS01_RAW_VIEW'] = {'owner': 'CS01','space':0}
db_list ['CS01_TARGET'] = {'owner': 'CS01','space':10000}
db_list ['CS01_TARGET_VIEW'] = {'owner': 'CS01','space':0}
db_list ['CS01_TEMP'] = {'owner': 'CS01','space':0}
db_list ['CS01_TEMP_DATA'] = {'owner': 'CS01_TEMP','space':10000}
db_list ['CS01_BKEY'] = {'owner': 'CS01','space':0}
db_list ['CS01_BKEY_DATA'] = {'owner': 'CS01_BKEY','space':10000}
db_list ['CS01_BKEY_INP'] = {'owner': 'CS01_BKEY','space':0}
db_list ['CS01_BKEY_OUT'] = {'owner': 'CS01_BKEY','space':0}
db_list ['CS01_BKEY_ACCESS'] = {'owner': 'CS01_BKEY','space':0}
db_list ['CS01_CORE'] = {'owner': 'CS01','space':0}
db_list ['CS01_CORE_DATA'] = {'owner': 'CS01_CORE','space':10000}
db_list ['CS01_CORE_INP'] = {'owner': 'CS01_CORE','space':0}
db_list ['CS01_CORE_OUT'] = {'owner': 'CS01_CORE','space':0}
db_list ['CS01_CORE_ACCESS'] = {'owner': 'CS01_CORE','space':0}
db_list ['CS01_CLEARSCAPE'] = {'owner': 'CS01','space':0}
db_list ['CS01_CLEARSCAPE_DATA'] = {'owner': 'CS01_CLEARSCAPE','space':10000}
db_list ['CS01_CLEARSCAPE_INP'] = {'owner': 'CS01_CLEARSCAPE','space':0}
db_list ['CS01_CLEARSCAPE_OUT'] = {'owner': 'CS01_CLEARSCAPE','space':0}
db_list ['CS01_CLEARSCAPE_ACCESS'] = {'owner': 'CS01_CLEARSCAPE','space':0}

# intradata database permissions
db_perm = []
db_perm.append({'source': 'CS01_RAW','target': 'CS01_RAW_VIEW','access': 'WRITE'})
db_perm.append({'source': 'CS01_TARGET','target': 'CS01_TARGET_VIEW','access': 'WRITE'})
db_perm.append({'source': 'CS01_BKEY_DATA','target': 'CS01_BKEY_INP','access': 'WRITE'})
db_perm.append({'source': 'CS01_BKEY_DATA','target': 'CS01_BKEY_OUT','access': 'WRITE'})
db_perm.append({'source': 'CS01_BKEY_DATA','target': 'CS01_BKEY_ACCESS','access': 'WRITE'})
db_perm.append({'source': 'CS01_CORE_DATA','target': 'CS01_CORE_INP','access': 'WRITE'})
db_perm.append({'source': 'CS01_CORE_DATA','target': 'CS01_CORE_OUT','access': 'WRITE'})
db_perm.append({'source': 'CS01_CORE_DATA','target': 'CS01_CORE_ACCESS','access': 'WRITE'})
db_perm.append({'source': 'CS01_CLEARSCAPE_DATA','target': 'CS01_CLEARSCAPE_INP','access': 'WRITE'})
db_perm.append({'source': 'CS01_CLEARSCAPE_DATA','target': 'CS01_CLEARSCAPE_OUT','access': 'WRITE'})
db_perm.append({'source': 'CS01_CLEARSCAPE_DATA','target': 'CS01_CLEARSCAPE_ACCESS','access': 'WRITE'})


In [19]:
# loop through each databse in the list
# create database if not exists with owner specified as project_connection['username']
# and space specified in the list
for db in db_list.keys():
    sql_create_db = f"CREATE DATABASE {db} FROM {project_connection['username']} AS PERM={db_list[db]['space']}"
    print(sql_create_db)
    owner = db_list[db]['owner']
    print(owner)
    sql_give_db = f"GIVE {db} TO {owner}"
    print(sql_give_db)
    with conn.cursor() as cur:
        cur.execute(sql_create_db,ignoreErrors=[5612])
        cur.execute(sql_give_db)


CREATE DATABASE CS01 FROM demo_user AS PERM=0
demo_user
GIVE CS01 TO demo_user
CREATE DATABASE CS01_RAW FROM demo_user AS PERM=0
CS01
GIVE CS01_RAW TO CS01
CREATE DATABASE CS01_RAW_VIEW FROM demo_user AS PERM=0
CS01
GIVE CS01_RAW_VIEW TO CS01
CREATE DATABASE CS01_TARGET FROM demo_user AS PERM=10000
CS01
GIVE CS01_TARGET TO CS01
CREATE DATABASE CS01_TARGET_VIEW FROM demo_user AS PERM=0
CS01
GIVE CS01_TARGET_VIEW TO CS01
CREATE DATABASE CS01_TEMP FROM demo_user AS PERM=0
CS01
GIVE CS01_TEMP TO CS01
CREATE DATABASE CS01_TEMP_DATA FROM demo_user AS PERM=10000
CS01_TEMP
GIVE CS01_TEMP_DATA TO CS01_TEMP
CREATE DATABASE CS01_BKEY FROM demo_user AS PERM=0
CS01
GIVE CS01_BKEY TO CS01
CREATE DATABASE CS01_BKEY_DATA FROM demo_user AS PERM=10000
CS01_BKEY
GIVE CS01_BKEY_DATA TO CS01_BKEY
CREATE DATABASE CS01_BKEY_INP FROM demo_user AS PERM=0
CS01_BKEY
GIVE CS01_BKEY_INP TO CS01_BKEY
CREATE DATABASE CS01_BKEY_OUT FROM demo_user AS PERM=0
CS01_BKEY
GIVE CS01_BKEY_OUT TO CS01_BKEY
CREATE DATABASE CS0

In [23]:
# loop through each database permission in the list
for perm in db_perm:
    access = perm['access']
    if access == 'WRITE':
        perm['access'] = 'INSERT,DELETE,UPDATE,SELECT'
    if access == 'READ':
        perm['access'] = 'SELECT'
    sql_perm = f"GRANT {perm['access']} ON {perm['source']} TO {perm['target']} WITH GRANT OPTION"
    print(sql_perm)
    with conn.cursor() as cur:
        cur.execute(sql_perm)

GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_RAW TO CS01_RAW_VIEW WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_TARGET TO CS01_TARGET_VIEW WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_BKEY_DATA TO CS01_BKEY_INP WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_BKEY_DATA TO CS01_BKEY_OUT WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_BKEY_DATA TO CS01_BKEY_ACCESS WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CORE_DATA TO CS01_CORE_INP WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CORE_DATA TO CS01_CORE_OUT WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CORE_DATA TO CS01_CORE_ACCESS WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CLEARSCAPE_DATA TO CS01_CLEARSCAPE_INP WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CLEARSCAPE_DATA TO CS01_CLEARSCAPE_OUT WITH GRANT OPTION
GRANT INSERT,DELETE,UPDATE,SELECT ON CS01_CLEARSCAPE_DATA TO CS01_CLEARSCAPE_ACCESS WITH GRANT OPTION


In [9]:
sql = f"CREATE DATABASE CS01 FROM {project_connection['username']} AS PERM=0"
with conn.cursor() as cur:
    cur.execute(sql, ignoreErrors=[5612])
    print('Database created')
    cur.execute("COMMENT ON CS01 AS 'Base database for Clearscape'")
    print('Comment added')

Database created
Comment added


In [10]:
sql = f"CREATE DATABASE CS01_RAW FROM CS01 AS PERM=0"
with conn.cursor() as cur:
    cur.execute(sql, ignoreErrors=[5612])
    print('Database created')
    cur.execute("COMMENT ON CS01_RAW AS 'raw database for Clearscape'")
    print('Comment added')

Database created
Comment added


In [13]:
sql = f"CREATE DATABASE CS01_RAW_VIEW FROM CS01 AS PERM=0"
with conn.cursor() as cur:
    cur.execute(sql, ignoreErrors=[5612])
    print('Database created')
    cur.execute("COMMENT ON CS01_RAW_VIEW AS 'raw database for Clearscape'")
    print('Comment added')

Database created
Comment added


In [2]:
def create_db(db_name, owner, space, conn):
    sql = f"CREATE DATABASE {db_name} FROM {owner} AS PERM={space}"
    with conn.cursor() as cur:
        cur.execute(sql, ignoreErrors=[5612])
        print('Database created')
        cur.execute(f"COMMENT ON {db_name} AS '{db_name} database for Clearscape'")
        print('Comment added')

In [14]:
sql = f"CREATE DATABASE CS01_TARGET FROM CS01 AS PERM=0"
with conn.cursor() as cur:
    cur.execute(sql, ignoreErrors=[5612])
    print('Database created')
    cur.execute("COMMENT ON CS01_TARGET AS 'target database for Clearscape'")
    print('Comment added')

Database created
Comment added


In [15]:
sql = f"CREATE DATABASE CS01_TARGET_VIEW FROM CS01 AS PERM=0"
with conn.cursor() as cur:
    cur.execute(sql, ignoreErrors=[5612])
    print('Database created')
    cur.execute("COMMENT ON CS01_TARGET_VIEW AS 'target database for Clearscape'")
    print('Comment added')

Database created
Comment added


In [17]:
# grant intra database access
sql = f"GRANT SELECT, INSERT, UPDATE, DELETE ON CS01_RAW TO CS01_RAW_VIEW WITH GRANT OPTION"
with conn.cursor() as cur:
    cur.execute(sql)
    print('Grant created')

sql = f"GRANT SELECT, INSERT, UPDATE, DELETE ON CS01_TARGET TO CS01_TARGET_VIEW WITH GRANT OPTION"
with conn.cursor() as cur:
    cur.execute(sql)
    print('Grant created')

sql = f"GRANT SELECT, INSERT, UPDATE, DELETE ON CS01_RAW_VIEW TO CS01_TARGET_VIEW WITH GRANT OPTION"
with conn.cursor() as cur:
    cur.execute(sql)
    print('Grant created')

# dbc access
sql = f"GRANT SELECT ON DBC TO CS01_RAW_VIEW WITH GRANT OPTION"
with conn.cursor() as cur:
    cur.execute(sql)
    print('Grant created')


# val access
sql = f"GRANT SELECT ON VAL TO CS01_RAW  WITH GRANT OPTION"
with conn.cursor() as cur:
    cur.execute(sql)
    print('Grant created')


Grant created
Grant created
Grant created
Grant created
Grant created


Allocate space for the databases

In [18]:
with conn.cursor() as csr:
    # drop space database if it exists
    csr.execute('DROP DATABASE CS01_SPACE',ignoreErrors = [3802])
    # create space data with space
    csr.execute(f'CREATE DATABASE CS01_SPACE FROM {project_connection['username']}  AS PERM=200000000 ')
    # give space to database
    csr.execute(f'GIVE CS01_SPACE TO CS01_RAW')
    # drop space
    csr.execute('DROP DATABASE CS01_SPACE',ignoreErrors = [])

In [19]:
with conn.cursor() as csr:
    # drop space database if it exists
    csr.execute('DROP DATABASE CS01_SPACE',ignoreErrors = [3802])
    # create space data with space
    csr.execute(f'CREATE DATABASE CS01_SPACE FROM {project_connection['username']}  AS PERM=200000000 ')
    # give space to database
    csr.execute(f'GIVE CS01_SPACE TO CS01_TARGET')
    # drop space
    csr.execute('DROP DATABASE CS01_SPACE',ignoreErrors = [])